# YTP E2：在 Colab 重做同一個修改（第 12 組 yTp）
2026/10/17，CPU 即可，無須 API key。先以「檔案 → 在雲端硬碟中儲存副本」保存。

從上到下操作。這是第五篇的課堂改編練習，不取代原 Lab 11–14。
VS Code 使用本機 agent。Colab 使用儲存格提供檔案內容，再把提示詞貼到自己可用的 AI 對話工具，將回傳的程式存回檔案。AI 不會自動看見你的 Colab 檔案。

**先自行完成，最後才對照 solution。** 程式起點為官方參考倉庫 v1.1 的 lab10 快照。核心任務是加三個單字和 countDue，兩個環境做相同驗收。


**第 12 組說明**：本筆記本複製自助教的 `YTP_E2_Colab.fixed.ipynb`。第 4、5 格已經填入本組在 VS Code 完成並驗收過的 `words.js`、`srs.js`、`test_count_due.js`，內容和 repo 裡對應的 commit 一致：`1e64db2` 改單字、`11836a2` 加 countDue。四選一作答等延伸功能在 repo 的 `app/`，要用瀏覽器驗收，不在這本筆記本裡。

In [ ]:
from pathlib import Path
import base64, io, zipfile, subprocess, shutil
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None
IS_COLAB = colab_files is not None
BASE = Path('/content/ytp-e2') if IS_COLAB else Path.cwd() / 'ytp-e2-colab'
print('Environment:', 'Google Colab' if IS_COLAB else 'Local')
BASE.mkdir(exist_ok=True)
# 重跑會還原程式起點，會覆寫此 BASE/app 的修改。完成後請先下載成果。
zipfile.ZipFile(io.BytesIO(base64.b64decode('UEsDBBQAAAAIAMEBUF2XUPcx9QAAAEEBAAASAAAAYXBwL2NoZWNrX3dvcmRzLmpzS87PKy5RKM8vSilWsFUoSi0szSxK1VDS0wcL6WUVK2lacyWDFRWnpuYB1eSllisEp5ZoAMXT8osUNCCS5Qr5aRBjNKu5FBQy0xQ0FMv1gBpqahSAjKoMTYWSjKL8crB216Ki/CINpWdrWp7s7X2+Z9ezPf3v98xSUtBW8Ar299MrLinKzEvPTKvUKNcE2gIxDWS7XkZisQbIVE0spr1s732xpP3ptHVP106HmgZWCjIArDkxJUUDKlIL9lJ+TqpeTn66hhLMrwrPVi16Nn/py4ZZLxv73u/pedq6UQFsDlg+JzUvvSQDyFVSAFqh8GjOZFDYAABQSwMEFAAAAAgAwQFQXd0bo6SzAQAA/AEAAAoAAABhcHAvc3JzLmpzTZBdaxNBFIbv91e89qrBJWnUVkkIkgsvhBjEijelF0l2QtY2M8t+QEqorK2QlFgidiMmVdCosR9iFQoGuvkz2v262r/gWSJFGIaZM2fO+7xvJgNDN9JPDfy2HVoIzh2vM/PcfuyOLqf7wfhHMDiN3o//2M/pGnZOvF8/g3c9f7gTvXobuy/9Pfvy4ovXP6Pdn34LjydSJoPY7ZYqVdzGqqar3MTN2D18WCqWEU56ULnCWumG2dxE9GEnOD0Ljy7Cr59id0+qCW6YuF9+fO/Rk2JpFQWsLcnIyrgh45aMO3ReWc8DII3g8DWyz5YRjF5Ebw6ikeN9PvYHU5IOzseeO8ASEgPOUTJYqlu8ZqqCw6g1mGJtssWqaMkQGzJMoVS2Um0JmMvzKumKDdzFg4rZSDdVnvTiesKxnEIO2Tz16sy0dI426C1Hf2QoFsvNh1HvlYc1Xl3Hdl7alpJgwu8HKAuFEeU8LYILu53A7oeTmTck1m5R06gY7c7C3RnRR/bQs3v+ieN/7EpqHYvmlsZEHU2RuMC1QgELFiVaVzlTFlL/6mnW0oRuGuSkfWVZ/i9ZIvoLUEsDBBQAAAAIAMEBUF0CXTQnggEAABECAAAJAAAAYXBwL3N3LmpzdZFPSwJBGMbv+yle9rQLuf6BTmIgInjwFtEhOqyzY7uxzpozrkkGURRqhSAllQZB0CWK8BBGRt8l3NWbX6GZVcEOzWXeYZ73md+8D3IIZZBKpjJpSIBcZcWQ6yA9VzaMasiNyXEQKxyGrJ6DaAT8q49xoz4d3rlR+DlrgxubDi8m9ab3/ey1On7j0u+d+K2u33mTUGC9nklns9x6S9bC8grIFjHwvmaygi1OFadkUG2XipqWFhWi7rwq6MTKY8r40SFBO3fVimRH3o4vY3nNB5h1TYfdhQYmT+eQDkViq5JEsZ3XdMNIu5iwrEUZJrikcBrKdFugYEiswYEEgLWKbrENwixbQToyMdWcIiZKMCNVYyavkRAj4Ze0bSX4oqqq8dmkvNfG5PHevz32Tuvey81ocDT6/JIO1fi/FHnMkPmXoYRp0SHGpsVMRRIRzFEKOpcq4n6vzOcy5zEtJlrFVqtBYLekUQMqDnPdmsXEI/P7bb/X4HmNBs3xe58/wfkE4y9QSwMEFAAAAAgAwQFQXStSpxW/AAAABAEAABEAAABhcHAvbWFuaWZlc3QuanNvbqvmUlBQykvMTVWyUlCKDAlQCMtPTnQqTUmpVHjR3PN02rqna6c/XbL0yZ4tSjogpcUZ+UUl8TANcMVQyZJEoGRpUQ5ITg8ilpJZXJCTWAkSAUrnpSTm5OelQqSSEpOz04vyS/NS4pPzc/KLQGqU00zSzNJMIQpKMlJzU5HkDNLMzcygmjOT8/OKgaLR1UCDi5JB8iAhvYK8dCUdoFBmVSpIWsnU0KgCiEFiJZUFYFdn5iamp+qDFCrUxnLVcgEAUEsDBBQAAAAIAMEBUF0Qct062RkAAEA3AAAOAAAAYXBwL2luZGV4Lmh0bWylW3tTFEm2/59PkdPune0O6abBxyoPbygyO97V0avuTmwYExNFd0HX0t3VVlWDyBrRqMhb0AGRhwr4wgcCjg9E0O8y01XV/Zdf4f5OZlV1dQMzs3HZjemqrMyTJ8/jd06eTBu/CoerGDtz8uh3X7bmTkqtrDb6ZXPGnt4sLCwVBvrtyUXr8fIvuQn8/2zL0eOnWlh+a878qdccufNrrncfMwduFMfvsuNySsW76G6N3LIGVr9sDph3XpnLU+ZIH2s+9w9mjqyYfY+/bA5iQmty2ro3jpkShpHR62tqjK5wt5GJtCtGItsaUdSaTjUmtWbj8e6aPTVJqZXYGuisi+BnpC5adzAcPRyOHgIxvJt9T81XH+y+p2x/5AAD10wshBXnezGXvbhReD5qTT225h5gRj66Nhqu3Q+K9sQa++f5Myy//rHw6YnoaX4aMR+8LyyMCFYLb98XP97HQCJ6mFkD49b9ucKj/sLDLXNglYE3XTAYbtekeFYyFDVd82VzNqZEulNJZk18YCnJ0JRLzBwfsaavWusvaVD0MOeUj6zCNKw2whRdz8rM7Htmbr7FhHvqDv+aG6kUO40icbNC38/W3Mtfc6MQ/J59UXQtF//MTuJHdywb3Zm58sJa77MezxSfTVkT76zFe/mP77DkX3NXiZ26CDtzltWic1yN6STlgRvWxIpguIbaajRZiqdkfIolVV3WGfjlCpmh8fgTnEdScQbd5NevWa/e2m/HrafDwp7y6zlz7HHh2Zr5aia/Ras4qXTKzH732ryX+7I5bA08N+fu4KH5BDM/bdkv5tHFHHpQuLYFxrE4vFpDucLCTxCJdf8dCJPNzc6aa4/YKVlLSUrc4UTQKc6s2U97xbR8ob2Qnzm3KrSdXx/Ef9F4VsWypAyeIPNCrg/UsXbzPcbMkg1YI/3m5qQ18cm602/NvXDkhb/jLadOY7XkP44tQt3FOz/BlSDsYm4c2rCXPhPhz/fMW1uY0RqexWtxfty+sSbU5S0tv3G78GwSfczJsdIcQihMj2mynNYTqhHOJNS0HMmk22E4A/mPj+3rC8X5Zdac0NSUkk0x5Qx1YIUnd8yxh/bEoj0BLxgRBm+OrVhjs/bcA+vZA3vyRfHeojPTPq79OsizTZaMSu1TW1aTa2J6Z1hJZVTN8BnBvqjwSth54dkT8MQ9ybUJDIn8S2f2mwmzf8vcHGMZSdNl2GfQkC8ZIUx47PQprL757MlvSN2bk4XpB+Y4pDeFB4KYZxtCTeC92LsO2WK1mBFacqbolJJKXDLk71Utrgc1tUuvZvIlRTeUdDtNYL3+bE8ssVhCjnX82EWdiKPCkzFzsB+MWi+vwxrz66OYAJORlTyfAMSQkU0tWnODjlvd/Fjshy+NQK9gyJy9b86vldwHf4asGz8664UYYALEdd+A/aYPEGO/m4c9wbQL/c/N4cnC+1ulkX8/AT7gz8JlxXxw3OLAJPvlxm1GkpLgfPxF9CG/uj8nWCkurNiPNkAclAtLVx12l++yJCA1ec5QNaldZp2tYb52rKCwOQgSsNX81ls8sO9Pnz1+DuMdbsAIjLI4/chDELhQcfSt/fQjGsG13oUlguPmo83ftnDA66zDUs9923LyJDOH5l2lE3IpMTVNxgo9tISjdQfKJAaGrYFbTJM7FbmL2ZPT5twS2WF+fYhM8MZo4cWwY7jjA2DWGbk/wgSGkXLnXtq3V83FawKJ0mpcZsISsaBK1yi8XRVrst722kscPcYHrJ8nySVHVgrPHwlLg56B95414gE2KgAEn6w7q9bkG3v8Rv7jKPhFo7MaLu6EqhusOLtov5+h7zCpiSUHRw582bp/mMFwrOV1ZyUHsJK5nFgimeLgMByzeGeY1ewcZljh9aYA95UnWCUI88hg3bxtv39D8PJygYR+OMrsp7fNq9PW2Btza80mO3e1KyRJMe/jBjsrJ2VJh3HURgj5z0tau2w0AUjTJT0djEDF78zVT2CQLOzZpLm6Qd4zOwv8MMdvWYM/AS2At+bYqJW7B7snNVLIIStdGIBeC0s5Mv5vW5r/dvLEufMcNAdETpDfmCjO92E0md3KmJkbZobUzvIfH8LV8x9zxY1NWCCkVlx4ihauZnctPCqdPHrsx7/+/cRxEXhevqREwByYIvd99bYwvoXu1vwHJzjxTIGTmW05e/bo+aPMfHIVPi7WWux9VuwbJUVwgXKzuA4BCnS2Z9dFIDbXXuY3NuyZ6+hjf74NU0Va4dhp74wIMzDqDEL20l1QwGTF/hEYnL14iy+O4MgROX7A7vvXJDKRcEAv9othR/7h8JGqxq+On24+/88zLSxhpJJ4px8gc7q9KXA5Ef5WShsBakV0xk9KNiRgHaGs0RT4+/lvwocCbnNaSslNAXI2wvAAg28achrdupS4kWiKww9jcpi/VDMlrRiKlAzrsGq5qZaIJJV0BxaWbAqkpLTSBrwLsIQmt5Xe4fZqumI+IyGn5HBMTaqab8o90ba/HDwoU19DMZLyEUrL/gEXOkZJYGONaKxq1I1u+mWsVY1397RheLhNSinJ7nq9WzfkVDirVOtSWg/rsqa0NbRKsY52Tc2m4/V72va3HWw70JCCVSvp+mhDRorHERPq6/ZnLjUQqIYROdrT9TEwJGtXMEeitofzWe9wR217dEMydLf94MGDDZwJXbks10cOH9DkFO8Vk7R4T0q6JMRXv29/FJM4U9OETMoaKjuERpeNfQfRynkpY7oNiwBQy1oYfq9k9fpa6uJYPInhUlhPSHG1qz7KiG4tkdHaW6VgtJr+F4keCvlYrAODDVkIBwJKyjGjPo3sgHOckqG0dHv5in0jayN8bEpJhxOy0p4w0HJQrLY1axhqusdZHhbFiImShGujtLQ68O0jR8TEyjgPFas85F8k80vEYcxhk+QTy2o6njOq4ipOMBTpkrR0j39sK/SwT/L1aCeA7qkUeLkIHCZrsQZdRW7B/NaA2NYjVFwbjf5XQ6XGXUnVldQfNtRMvV/xhxy51CvpBIzW2GZQniC2c9La2rqD4LhRKJeJuPMRLZxdka6d0tt3s81Djmm6Mti3b992fnzOkpTbjHLKLCJrmmtFrsAbaxzHbaxxgIn8l2CqttLT0VLVmGFKvCnAXS1wpLEmg6a40skbybMCR7CBaKxBk9fXMd/Aka/TrXqmQYwRnyQlwPj0BEx/aNUHDhzwTUtw0ygMhtNDgiJLyQBT07GkEutwG4KhwBGRfNnLEwh0jTViTOVwtcM3FAx0BA0tK4eAnUo8LqePIEqYn6bYL3M/7UYhrVZSaJOSOpGIJSVdB4DD8D165ue7+Y0boDe1Gz1JOWakfSQlpeWSlMok5WCJq6MnWP4TYvrjEhFXAaQZT76eiZPJB/wzeeR19DL+N6tcJokVcteQ9Jmba9hll/Hn6vsiOv6n+oPVkv4Ee/8ZlzSlsGUuE0egHCd8EjLU9vakfIL3o1X4U/UdV+GQVC95AuVO3egl89y09c4AyySlmJxQk3DbpgBSIL43zSERQiIip6svJ77eUxttkDJQT3Xh7jDyft7w9cWsajS00xY6mxEv1bRJfXidJOFO48zqGDV/rtSO4JS2ZN66/Cvyj+E8I2XU/KIStueR49+FoOQ4kfRvJsoJO5pyOfTJDbDi6dPr5/5gM6xkDKZrMUzu7Oeot2iv6KBrv/lZbFV2+HykqqaGeRUVZGjW+Euq/7gTInc7l9EQgNh+Kr30LRVnhsy+Nfv9Et8jvmZ/VQwqKHEyy+vIE+03OZ4Mzphjz5FCWnc2i7lpc2y18PCevfWKdtCDOeznocOynRsmKrzKUUHh2hbVdgytuyYmGbEEUU/KBjq3sybWc6WhCp9Yj/P+P+dOfxfhm+2gn1oECf4JpE3BADaE6BkIhdi//02j2RXGyQblUM+VqrZsOkZbDqZLnUCFHubS9pHSK0hVi0l1A2JpV9q6MXF7KFROmF3hEhEVIeYV6bBuqme9fGl++GQ+eoaM3766AeljZ07inL4uKivWVB+8w6k0TC85ZLgU4lL36bY2sIS1R11R+Bv37ioG9ApwKUQrhIBkFbs5Q0UHEAiGWNMRdkoyEpG2pKpqweOSIUfSahc+1LBDB/fLB0Jsb2nOhqouJY08LdLZel6DFJOgkSYSPcz3B2E0Yxo1KbPC5jptx93uwdoQM0d/Nj+/KGwNEiI8egZf+B090Fqqfevey9IVGqgSO1VSbgS5vCrBSRuqYD0+nSMMy6fbgl2hHvQWUtDBPfR5oSsip38gYQVLb7A+pssycuxq1qWpyEKi7EqIshiljQX1CJIR1tTUxJBvyW1KWo7DnpxWVtuAx3hWxiMXdJCzy+VSGBgSPiNKJuZov/2G6kqabGQ1cAmuuTGFvT9WWREVxZKZ6xWlEb/j4WuFv+1cIPFNU5IUia85qxtqKshlJbTjMPhHXJDDieOEF34o05RLhjdXVbikM2lS0Y3fd04xyXb3pMHb/ZOvNpLJ6olgJBLxrxB9/XYrxGLdXXFMpDj9EBtvqtyMrcChGZHgpcfRwrUtp3Iy0m++mqniHnsxK3O1i/nalCRy+WAXOUjJ/rhlNHqmwRkgQB17XnjUD0ylotnAKjCEEGPmujm6wEljg0DOlk0mYdaCuT+hAQk0qMfVWDaFLR/poSUp0+Ox7hPxoBIP+Z0gjQgqlPqnoJMwhSIilJOtIoMrl4YTDeqofDA2C05gP4WpMXP2PkMqVRygKhNCof1sQ6RVLmFQpWDdLDbGIB0IuJ7DJRRJyul2I8EdKMr54QN5VrxtaP7jEGDCen2b6pZcMEIqAiituzfxtfAEmUVvoMGl5CbSoYiSTsvat+dPnSRSIq0udXOSYJ8M0IjUtrwBmWqllDgBYcv0TAAk9MOXdyH6Q0PV7itCT0CM0+P3Od2RT54rO1//IMN8I8Jx0WcRbtbfU8nNdpYvJ3blxp3i95nx873NAMXHXQzQ/vyRanOfkFKM2m+fMC+XL1sP30ioHRUg7zoflsEhXI8QtO/d28BN8iv0RxNHebR5I0mfeiwhx7PYRHBsr2ZqR3Wl4/5yY5xXK5eG8xuPwJl3LsAnEhFBo99SVNDolzPCMxHXN4gP4R96QmkzxAcZMnFaOX6VdfDAg2+P4BdgBD4iXIayNypxjtjLFGvz68NUfs5dw7P1cgGkBRyUG4RjJY49iM3rTi6JNCe/1c+xccYcfMYCCMtlvr0XTebm28AOEc2nVU+L/JgGiM3s2TfmrU129MwJfnpF5xx0gIVwuf6u2H+reGuVqtifX5hjdHpbnB4tztxE2ugPZUJ9p04fb+G+hJw+G5fDugoHM8IHwgcCjtREtZaVf98fPkDFy+K9+/bkO3NwNL9+J//xIdXVh+8Wnj9iLeFo9C+iZGvN5YD/5viovbQGBEJKh8DOjjnzYwU//qPl7LkTp78jLuqidfvC0YPhaC2fnuaX0kZCUzNKLNwpazpJ31qaLs4vm7lNitg+iiX9AOD/JncLBVFY6PCCghupO3hWs0ts7pCRF25LnrgXdDhATOMzmprKGMFA4fVmfn0ovzVPGQUWxDCeRIFsf/muObeEL6SZ92tUOc6NFZ5sIYWFcAIhAZG/E8SJm2rWsZ2hK6WUqMM1IDpl6B8kRpyCK4VpCpajOXNhiMrjC0v8aHiEgvLSgnkfdrWCME0HV4hYRkJJdyh0xsgHUBT/dI8Z3RmZYi/yK34oVa5oOgx2U2Z8PN0G2cQlQ+IxF78RhxU31LfSh9YIJ0rBLUCj4D8pKeN9ozPCyL9UJR0MkJiqSjBHuy5raMjzD0aLFV5hDW1ZA+O0GZtYEujinTZQ4vD8OR01bOsDcVhzzwmTlob5EKrqlyr5D+fo3IFEy28fuLcODjLz5gN7cdOeXTfHRyj3GX9hzs4Wpic9weY/v7ImPoBd4WTcJ+hg8/oC5JTKyBo/XKX22v1od2THJYrXlHTpR0PtkNM671FLQpb07nSMeWYOg0k2c78Mipp4NaOCrw/XNZmQXeqSFEPIKBhwL2JIGSXi+RZUlKrprK1JyboO+6OcUdh5SjYSaryeBc6cPnc+UM3bqK4HV6x3ujCAh1BwmFQaQGeqWShiq1HDDwaq3Z6XwpiXm3S956Xex22ujk4+gNihX1xKt8uamtXDcUWTY0a4lY6BZS0sxWJYCvFCYTfAkcTzPYGe5uc+c2i+AkOI/hUxEdUt6ytTZ3fJKTUuJ+sFfFazkq7qWW20bn91GXTui0ZhQp5nkb3yziz/4YH9agoIAdMqcymhbS/ACzrI7Px2U8+ikX10/aQSuBnybuvmY3Nsyt+dPH7uGrn4fmLHP9n+0mTCjOqdX1fgrlXUswvYnGDHCrGSocFKHM3Xc8NjV34QAgwROoVKSYJAA8cOYZP8tEhEbgfAhPEHqaN/s7HTnycQeJYfXpBQRnYBJ2Bjhev4Cp8lb6Htkpd6VjORlTi5eoMTSmISkh2n7OJCt9v2R/Z9RGvnyouIMJzWBeLkh5C34U2oXWUfGnyyEFGe3zJZpoIKNl1Da8WRRYr7/DhanEXzg7RLlTmKPfEAaIiRv+SeBLwFcSUKkcTKlyU06MOdnXZChNC7AbJ/bMgxrz8jQCJKFV/cBSOF4bXC8/sFZC/rdOOGcHtz3ezf4FyAcE9Ap1O7dAxAE8A2NVAduJzwHg0l4zxfcTagT3p5uhtXu9LMnBu1Xl7/s4cjxBWvDPDEjGufErJibl7UYVlzyzdn2bFaJ/uiVAsMLq8gBhR6V6xrP0OVIoTz3CAWcVmLGOpJtUvWmiXIjDYtsSQWrAdpivJvIVcIhGhdSDa7WIumqVowIObEjo6qkOMv7NlXyHsEu27i4DMJMts/lE3A/rbVAzidyoIAJyYsj0/nfGP46FjIDvaETNWaeUEFlHXnDoR7BWzU3ForPvgsdu88raDl5T8M040EZNwP56DtgLtPLGXagoHShGUbwEY5dYRUV5I8KbCxBs2NrZrz6TJPs+ldKA1GJD7AWMR8u+XevMDLzyus54v2m0Ugy9mjfw0nFUMWF5Gs6Q0LWUH/RnFhZMcyUZcsdci6ERRpo+POvO7BF8RTni5RGtyl2tZTUV+74jssdf4csj0MHVkXh67LCXq6nKhmQH/AtbNxYzVUlSOKRLo2xAiDhAVGdDrgCEqAvZDIwWgkCzOJP7idENblYJSnozugqu+8p4SqZBiuHPb5goIHr/zch3/YwaKEcH8Doai/F1t2RKZt0eOPZo/l+CQ4+S1kwtZON3ja4sCRmqFXpCEXAvy97D8/VP+Z7XWnQE6jAxT4SHzuSnSLxyuYySVLqTyQgP2IP+IbUGnd6Tf7hq3NMSpvEkJd3Rnz8KEEe95+EVszc329OLJCtSKOLI5vGCXDJHsi/4Htc5B03IlvX5wEHbgYCHlLCZBsPt2HxrBbEDgKIdFNMro8N1vsXccz1fZ714vzfeBV4Ju1/NBedM5T4Vl0HTI3jJ1V4fMt4u7Da+unT+hPIOGiH1f/xfIAxTcOPkw+qmlSd0TR+W/wYsTRCI/A3pu7G/8Km5H9+OJ52FelLh6GX4wIVSGKb8NsLJFwen4TxkOQNz6AZ5ff7eiV4Qh1MeIpmIMXWsWA0uRtqtYCjA6qGY4VDn+OG9Fuy61sxjQZzuoUN4MBcebmMsCcjVXJu9RM6Ytzjuedd/R4UhDzqB18AN+xuTJoqOjjopf7vXReUGpp2g5poRKd3yw67ebQsBqSPL8aSbfQZu97R27FO4P27dUSea+U5Cy7TCd7m2iV/801A5rm6ijVS7mK4JCedrYBsP+vno+GNReePnS8yV07ESD73UbQ5cfDdmILeyg5HW9OKMl4sNVh+cr/JxLzC/QDXs5jji6IYGyOPy2F3t89WNnxpv8AFmvdH4ffFp9NFZYH+dGAcwF45rpX66P7cLnpwsMFYKf9akFcRKZb19dfm8t3K+pTXlApP4Hv2R4/SifuXhDxqqVfld7cAqKvKcTLz3onoKwNLrSt8Os7IS/N28PrmzJ5vA5dNFXcQ/buOXuk8T0rh6pF3HeLkeKK8ZfNe6Vz5WGqwQyOCqF606X0dv8y6YCcLxPtu54fqB0OpDmJk//gyHesQzsnmBKVVB0DKz8CUjucZtBzIcjJVvj53m6nNezrrwGdvM5ZSn1DIX+BtouyTU8Wd1ZhVKIea66uifMLX16d1ThcVx6KlFbpFmlFHdit4Qvi5s0HdPv//WvH2l+NYAp79g1tkfs34AW8dvPEHOynAhfawczmW2ty3TtwKSnREzLlqHRQuTRk9ZJvMb8mnIviznXOEXIU2DxFMV5BIoaWHor7Q3xLMYqNeWHxhT35DvkHIySiLezqKP3jAtqfHxD7c2EOSSVNbPBAh+13Ug/508f4bwSDuNJJVc14hdXQm0BbTiwe4bc7vpNSNA2aGril+dEoTrrj+2HwEgwIe+VbKE8jorCNyC/2K3gt2SQBrIOMaCtLNMozCkCps8ESzubZoEz9xeR075cowTBIMDSt+GcIohES0FXkwwFQ4LT83l1xYaXk4THuHzyWlZzFgw/x1XOwCtes9EnnSI7LqNh708u0OOS6p2PMqy+DgTZFw1ZNXLrnayub0JOse8LtJm4P/JdNYFH5z/fsyWmeL4V8p3A+KLjwg6hqbLsUsFucRb5ory7QhGMrxXtU7v41N1LsH7VfTYljDl5ZnhEnvuIsnR8KP6ezD+8fYBDPy1OkCfdohUKOczeHTgvmXjiXzpc+mjeGmC5rdD2ZCHTIGlxDXJc+8/1R8guSWsDp8j3vEWBKmqWlTqVdwv43VHqMlHXDctsVnerRAf7vGghVfZeBapxrizXi7vX/AVBLAwQUAAAACADBAVBdmfaRR/EAAADHAQAADwAAAGFwcC90ZXN0X3Nycy5qc0vOzysuUUgsLk4tKlGwVShKLSzNLErVUIKIKGlacyWDVVQrFCdnpKaU5qQq1CKr09MvLirWyyoGqeSCaNJLSU0tCC4pykwucS0sTczRgOnUMNRRKCkqTdVRMDQw0NRRABqalF9hpWCko5BSmmoFFDVSqNW0VlDQ11d4vnbK0w29z2Z2Pt8yi6C5xtjNNYGbawEyl6AxptiNMYUZY2gGd97LBU1Ahz3Z0fu0rfdpbztBo4FOSUvMKYabDTXaEO5CQ2Sfv+xZ/6xz8tPZ857PnqRgCI6B/JxUvZz8dA0lE4WS1OKSYoUCkJUpCo/mTAYFPQBQSwMEFAAAAAgAwQFQXWJftLdJAgAAigMAAAwAAABhcHAvd29yZHMuanNVkz9v00AYh/d8iiMTSCVmYKhaZUBCYqkA0YEBMVycc3Lg+MzZDm0RUgMEEhLBQJIKl1RpqNrQqq0CKW5TQ78Liv/clK+A4/hSV7I82I+f931/r08QwEtCs1rqmQb+rTeCCzitI+dww6mXx7bplHvM/OiU+57VG9t1tv4T3MP62K4swQyYB8sqxYoObo/tzYdLd+4Df7cGsJJFK6m8XpAB67zxDo793rm/931sVxMiUTQdPH7w6O4ySIMnCQBeIWUhCeUcoVjPF5JzYC2/kHTthne04f5qJl/PgYihSDSohokSMay05V8MAoBLikSEGUOGdDUiJjP8+TpVTJECKgR11lAE+G/r/v5uTIEkCYsYKTrv41vVbVa8Tx88893EE0IqCtrQOOJ0ym69FIwf3L1hPSajSKcYFXkx53PL2dyagPtdb9ANwJAK0kO0CGU+VesLMxtTiyAAp9b0qj9Gp+dsux7bSoXHDpwdczQcTtczSc0aBEtyKu/dxjGQYebWvCAhqBsUCeGSb0IFF6CsTXbBZ5aRmoezkZ2dPb+/HYXGc8sSWc1jnrx7Yvl9Mx4bzMmzTNdLzDq7/J4rVKTkjJliZJfYSTumyGEKJYlLWNNinVN2djHzhJD2wsCUIp6V2/7L7E5MQkSdqIYWvfYOOuzQjHUSMs+hkoOUEN5ut3bVkTH0YCGSzH8ivz3w27+v9iFSIpIsns3M+lZYKfF0MYElcF1fVRGRQIFkDRmBa+k0SBrBmZCwgrLJG9HzFFpRCdW14ByE52Ex8R9QSwMEFAAAAAgAwQFQXTZybopmAgAAiwgAAAwAAABhcHAvaWNvbi5wbmfrDPBz5+WS4mJgYOD19HAJYmBgYgBhDhBVLeW8loGBI8jTxTGkYs7bu7sLRRxFHAJdTk7/dPCQutjJyUkTVsZ82Hgi3flA26Itn+/Ou2ir3mT26+vbmbNu7a2uqPzF5HD++kVuhgQeRoYOIYYGjVFqGFErfl/k9tixZ29/HNOc849ff36hWlNUzq/vv/j6B0vLL4nOSRvYGW/MWyxfpsR8W6hB1/LXQ7aUBG2mDbzu9ndm8Kx5rr2A1+Lvw29n7lYxbZgWy7F4xr6pe5iizj1rbv5glSHpz8Bt+fehxZnHRkwbWu6yuSdriQuArJ5x1pW572TOOVWGzzcSC1LUSsQYZxy81/TYePllTgaHKQw+T6MY0xOj1MHunHGklenc0TN+igwMgUwFc1ezzJl4cq8XA8MiFgYFMPWAa73gwz1n3h+HCYxS79z1G/5X19TK7WW5cenp29ovQj+mq5OZSj61G7A/vW/csLeSiY6J8kFEXd1/syrON6+Pr7+/ePqBwAribbdnOx7/J0iihAQtg5ja8ejj18ovQg2taCl83WcpwXMnnsSpgvKFQScwX3TPnOsEkk3QEuey4L5sNZGBwYXzAM860YKaZyVWjDOO32tSNl64meNATpmEzOSbc7zBijnFeS2kgnc2MOikbD9YOON46ju5uHlu6x/e0ct/v3g6Q2zfP1MbxjWp/w9qLMtXZbzREMuxPf3dtc9cGd5x7Ad5LX5tV2K4PZnxRufi+mimDeKSDbyWv74rMGxXYNvQ4Fyfz8Qg8FRbYcADcZSiNvWL+fGWGlv98LkFwEYGg6ern8s6p4QmAFBLAwQUAAAACADBAVBdAOgrOaAEAAB5DQAADwAAAGFwcC90ZXN0X2Nzdi5qc51XbU/bVhT+zq848hdA8sII+wTqpA5t0qRNk1Zp/UBQ5SW3xCN1UttAR4WUaoQkw4HSJtBQNtryVhivbVcQhPJfIPfa/sRf2Ll24hIGcYKTKPb1Oec+59znPsfu6ACdaPq9sDYc+E2D02Qev2AW3rCVLWbMsMzuWdKgs9t0a44aKei98wtQY4emVs6SOXN+nN9J/cEONq311fNShqYydOefs+QTPDE/pOgnw/z46rz0svd7sNIbdLJg7c+cl7It4bii6SBpGlF1uAUqeTgkq6RNcEeE9p6KxWNISKpGcFYRhqWYHJF0cjeuRjQYu+gW6HDhc8eWjg74Ag/P87w0z2bTiB9OJ54BzcwhTrYzjSfsxQ7bHC8f5+zikjNu2MVxOrVI1yat14ZVXESoPFSLCysQISRxR1flsP7twyEp1ladAXEnEjEiWi8m2d8LIeVXScGPaK8WrUJWaBdbAPoeQ0xWSDd0inBfJrGI1g19rpsgguB6Cv0wJkLVMnjR0g3JTStR0bS/vQe8A7Omi4dsYdMf7PnRSQ1eNaTwXw1qPnBj5F3NIv/mpx+RM70///Ad/pnrh7z6++/tJ1Pl4wl3Jconf5mFon9uRBFHoyHlYn6X0gj6pFGDrdEDc7Bev7VfbYEDAFz0/nABHAQgggsAoMmi3xSty3uazfEtun5oFo/LJ9tsKusLuVUYkIcJDCUEUWD5T2xpHLc3+7NgF3OYxEbOKh0IrXVy8LzhWvcm0uKsLxWQHzS1hmpkJ+c4Vw5yuLNpZsKezzeQjyb9DoIQlQVBEFn2OZt8SWdKdVJodRzQvpXnUHW4yWIgeju5ZB4aNLVOk5NuJuWjJaRS+SDpjTQnPyJ5pKtSMzTCM9en2RQ8HrHCARad7ubM/J6n0yi1y/7QESj04ayN7BVAoVI8B0/oaxoDqj1KuSP1jyE+2A30aYa9L7gdTASiqnEVq2Ann6Kw2MaOtbEMY9gRrNVpmk2zdydm/i2EoyQ8eG+Ex8OuUu0Cbksij2RNl5UB7D9YXqJ0gyDFBuKqrEcfYClHozjASnlzew6nFbCiPS0xooOK9jVAG+4e3ozXF0kVK8l6iCpL7KCpLLOr0M5tT5Od+5WZEOrn+vQhF3iNfWGHFA+4m8alBES6PI87s8E8AvHB+nRQAy7CwAMp0Ubg1tdAApzinBRIchR3bD5fXQgQH/R8+r7sD6hE0uJKQFbCsaEI0doE3HrIYQ7v/2aVvXCJ5qgtELwqfOcV4YlSE7rzUmgnOFEAJfiqiMErIo5GayIGr4qIHeiaiF2NlaDrc1QfDlSpL3qcr+2+IeX2havyodEUE+rS+eY06WqGIPvv2ELWjyBWJs3m3qChKzR2OmctpxsliWtdlyicgxt57HKuLQoWF9zMBF1eo7vT+MzNn6rrrtTt2kcin13WUPUv9QLO43/3eOeqIDLoMwOqKgp4w0p/qA8Sp6Urx1xCIiHlPi4Yv6zIhx9gZ61H+FqPBIjCHTAcb248jlALl3deR/uxCfBCHu/Ziyc09bF8NOv2Cxy/CbeCTmPijSIew9H4QJvA35j4G5aGLyQYMAKnC8/5e8p/UEsDBBQAAAAIAMEBUF0I82GiEQUAABYKAAAKAAAAYXBwL2Nzdi5qc41W21LbVhR991ds9JBIg0cOfYTSTEvbJ5p2ykz7QGhGYx9jFSO5khwTiGdMEmOTmEszEGroTBIIl5QQaJoUGuzwL4mPJD/xC93nHMs2NEAZRrb22Vr7ttaWIxGI2jfVn214n1vAf6CPXtKdJVrKQ9/AD0BLuzS/7i3f814v0EKVVuaOK0VvcdVd33FLv7rFvePKynf9n18Df+MB6EaMjKkJZzQJ9Sd3vO1df+vQ31w7rkyHIhGoHcx4q3vw5bfffMhNNm6SZlRLDjimpQ2T40rJnc7VDtfhmhkjQOd28bu38tqdXXcPXmBYh9jODZEsQqK7l5vzN6q0vFX757F/typ8vYWtD7k7IRbRvX+/dpDzn5ZEKcWCW5x3X9yrVWeOK8v13JJffkyLU/XlBSyivlKmlUVmKeVr1TmsWHjS/CZmQue3hT9GlSSoHa75T7cQm+YeiKcQoeE/PUPflbznb71ytXb00p2dDsXTRtTRTQPsVFJ3+nWDyEm8KBMhgKhp2A7EdZKM2dALg0M9aEwSB6JpC+8lKQy/pE2HxPAmriVtws7jpgUyc9LReqUHPz4FhqgmiTHsJNDQ2cnRA/xoAh2Zx6DOAwDocZAFcMNRmJhjby9cli7DpUuNJ6ATuoYCszLBM+vkNzxQD2TxYex2PbfmvS3R/PNmU+D91ENo71IjEsE6ToVTTpcJZ/+xyS78Tffe/Re0kVs0IarMXhDKsdLnRmKh/Df7dLM9/9OQUljCrogRqqm0nZAxC9Wx9FFZUXqag8Q+hT6eJ7Of9TgeWcRJW0bDoyeUFdR+VECV8gbT4lK9vIZXGJzgIwsHfMoOoRL88qI3PwVfcNn1fd//NX4gPVEVKCl//6/65GytOoW3WJ+X3/ReLqG40AcJXn+yA8QIjyeankzKTTqnNMsmqCzZIWNOG5stM9PiMjtTLZJKalEiR346PjyKhLEbisrFIEeuW1evGxFFRU5/pUUTsmxpmTDoCvR+BhNNqnagNehJoyHnzu30EEXBbYpoKq6lSQyhtMSh89leYTJozEaIi5s/QXMogBfHg1eGVMfsNzPE6tNsIiuCG8SQWhCDXR/1GU9I7UUxzvHe8wismYIWYrjdwAXZmjHPOdvOFPZIwBNGC0aSCTBHupEgOE4YT2SHwkAsy7RsZhOcsYhmm0Z2CLKMNBtzdLrgvjrCbYosJdGRGxnTwiaw1bvMgfPPceHV71b9jUn3d9zGK8RgKx6XpY+LdmkVjeJVUi/M+M8KyCj3jwXcpsJHGJFX7LY4RZ9t0r05urvNFzuD8t78yTZHw1iiD0sQZMBWil943SLiTS2pxzSH/MgcZFY+ljem245uDLfx0tFGELgXDJKBAeLIcuADt28jWxV1VEvJGUa8AaSaMSxnVGIoJycmJCnwzBHO8qCVTcrz9SxcTgsSzDgfj9Ki9kl2dTB24ToRkO2DDybUDRK2nE6v0PWqGEKg0VI9V0aWY+NBQo6cBO5EG/pKjCosf6yb7b5smyZ4c85gM2NN67Sr7S3SgT26IF8Edn/bZe/E5TMS4EDjiYuAMIf/AcTnrCY0W8bUmq+384DNEdU2R4kYPps6FyY+DVdBErQNCLvMWosOmE3zlxLdf8UlEJxlG3sEWumJ1xG/ivS0WIylJw4wfiMvoc9A1S1NM/02iZbl6ual3koRpNSoGUsnCSePlMYfYXGsLoZLRdhVMpYyLYcRdKK5tMMnVRNuLUIG/y9QSwECFAMUAAAACADBAVBdl1D3MfUAAABBAQAAEgAAAAAAAAAAAAAAgAEAAAAAYXBwL2NoZWNrX3dvcmRzLmpzUEsBAhQDFAAAAAgAwQFQXd0bo6SzAQAA/AEAAAoAAAAAAAAAAAAAAIABJQEAAGFwcC9zcnMuanNQSwECFAMUAAAACADBAVBdAl00J4IBAAARAgAACQAAAAAAAAAAAAAAgAEAAwAAYXBwL3N3LmpzUEsBAhQDFAAAAAgAwQFQXStSpxW/AAAABAEAABEAAAAAAAAAAAAAAIABqQQAAGFwcC9tYW5pZmVzdC5qc29uUEsBAhQDFAAAAAgAwQFQXRBy3TrZGQAAQDcAAA4AAAAAAAAAAAAAAIABlwUAAGFwcC9pbmRleC5odG1sUEsBAhQDFAAAAAgAwQFQXZn2kUfxAAAAxwEAAA8AAAAAAAAAAAAAAIABnB8AAGFwcC90ZXN0X3Nycy5qc1BLAQIUAxQAAAAIAMEBUF1iX7S3SQIAAIoDAAAMAAAAAAAAAAAAAACAAbogAABhcHAvd29yZHMuanNQSwECFAMUAAAACADBAVBdNnJuimYCAACLCAAADAAAAAAAAAAAAAAAgAEtIwAAYXBwL2ljb24ucG5nUEsBAhQDFAAAAAgAwQFQXQDoKzmgBAAAeQ0AAA8AAAAAAAAAAAAAAIABvSUAAGFwcC90ZXN0X2Nzdi5qc1BLAQIUAxQAAAAIAMEBUF0I82GiEQUAABYKAAAKAAAAAAAAAAAAAACAAYoqAABhcHAvY3N2LmpzUEsFBgAAAAAKAAoAUAIAAMMvAAAAAA=='))).extractall(BASE)
APP = BASE/'app'
print('工作目錄:', APP)
print('來源 commit:', 'badb04df66f77458a284b40d62bd16efb41f9b93')
print('檔案:', ', '.join(p.name for p in sorted(APP.iterdir())))

## 1｜Node 環境檢查
原始測試是 JavaScript，需要 Node。缺少時下一格會在 Colab 安裝；不需 GPU。

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# Refresh the Node.js path for existing Windows kernels.
if os.name == 'nt':
    node_dir = Path(os.environ.get('ProgramFiles', r'C:\Program Files')) / 'nodejs'
    if (node_dir / 'node.exe').is_file():
        os.environ['PATH'] = str(node_dir) + os.pathsep + os.environ.get('PATH', '')

if shutil.which('node') is None:
    if not IS_COLAB:
        raise RuntimeError('Node.js not found. Install Node.js on this computer and rerun.')
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'nodejs'], check=True)
print(subprocess.check_output(['node', '--version'], text=True))

def run_node(name):
    r = subprocess.run(['node', name], cwd=APP, capture_output=True, text=True, encoding='utf-8')
    print(r.stdout, r.stderr)
    assert r.returncode == 0, f'{name} failed'

for name in ['test_srs.js', 'test_csv.js', 'check_words.js']:
    run_node(name)


## 2｜帶行號的程式碼
下格輸出可貼給 AI。先要地圖，再追 mark(true)，最後問 today。每次回答抽驗兩個行號。

In [ ]:
for name in ['words.js', 'srs.js', 'test_srs.js', 'index.html']:
    print(f'\n===== {name} =====')
    text = (APP / name).read_text(encoding='utf-8')
    for i, line in enumerate(text.splitlines(), 1):
        print(f'{i:4}: {line}')


## 3｜相同的提示詞
AI 收到的是這裡的檔案快照。換檔後請重新提供內容。

### map
```text
只根據目前工作資料夾的檔案回答。先不要修改。列出主要檔案與責任、載入關係。限定 app/，每個判斷請附檔名與行號，區分觀察與推論。
```

### flow
```text
只看 app/index.html 與 app/srs.js。從使用者按下「記得」開始，依序追蹤 mark(true)、schedule、紀錄寫回、save、佇列更新與 next。每一步附檔名與行號。不要修改。
```

### design
```text
為何 schedule(box, ok, today) 由外部傳入 today？請用 app/srs.js 與 app/test_srs.js 的證據回答。哪些是程式事實，哪些是你的推論？
```

### words
```text
先讀 app/words.js 與 app/check_words.js。新增 loop=迴圈、variable=變數、debug=除錯，已有就跳過。只改 app/words.js，保留既有格式與所有單字。先列計畫，等我回覆「執行」再修改。修改後執行 node check_words.js 並回報實際數量。
```

### count
```text
在 app/srs.js 新增並匯出純函式 countDue(states, today)。輸入是陣列，每個元素都有有限數值 due，today 也是有限數值。回傳 due <= today 的筆數。不得改輸入、不得讀時鐘、不得改 schedule 或既有匯出。每個新函式前加 // 用途：。新增 app/test_count_due.js，測空陣列、過去/今天/未來、全部未到期、不改輸入、同輸入重跑一致。先列計畫，等我回覆「執行」再修改。
```

## 4｜修改單字
把 AI 回傳的 **完整 words.js** 放進下格字串後執行。請先移除 Markdown 的程式圍欄。

In [ ]:
updated_words = r"""// words.js —— 單字包：全隊共編，進 Git（Lab 8 Sprint 4；PLAN 見 index.html 頂端註解）
const WORDS = [
  {en:"algorithm", zh:"演算法"},   {en:"recursion", zh:"遞迴"},
  {en:"vocabulary", zh:"字彙"},    {en:"memorize", zh:"背誦"},
  {en:"efficient", zh:"有效率的"}, {en:"persist", zh:"堅持；持續"},
  {en:"retrieve", zh:"取回；檢索"},{en:"interval", zh:"間隔"},
  // 動物主題單字包（Sprint 4 多人共編演練，分支 lab08/feature/words-animals）
  {en:"elephant", zh:"大象"},       {en:"dolphin", zh:"海豚"},
  {en:"eagle", zh:"老鷹"},          {en:"penguin", zh:"企鵝"},
  {en:"giraffe", zh:"長頸鹿"},      {en:"squirrel", zh:"松鼠"},
  {en:"octopus", zh:"章魚"},        {en:"kangaroo", zh:"袋鼠"},
  {en:"butterfly", zh:"蝴蝶"},      {en:"crocodile", zh:"鱷魚"},
  // 程式主題單字（E2 AI Native 工作坊）
  {en:"loop", zh:"迴圈"},           {en:"variable", zh:"變數"},
  {en:"debug", zh:"除錯"}
];
if (typeof module !== "undefined") module.exports = WORDS;
"""
(APP / 'words.js').write_text(updated_words, encoding='utf-8')
run_node('check_words.js')


## 5｜新增 countDue 與測試
先確認計畫，然後把完整檔案貼到下格。保留 schedule 及既有 exports。

In [ ]:
updated_srs = r"""// srs.js —— 純函式：不碰畫面、不自己看時鐘，所以可以測試
// （Lab 7 Sprint 3；PLAN 見 index.html 頂端註解）
const INTERVALS = [0, 1, 2, 4, 8, 16];   // 盒 1~5 的間隔天數（索引 0 不用）

function schedule(box, ok, today){
  const nb = ok ? Math.min(box + 1, 5) : 1;
  return { box: nb, due: today + INTERVALS[nb] };
}

// 用途：計算 states 中 due <= today（今天或更早到期）的筆數；不改輸入、不讀時鐘
function countDue(states, today){
  let n = 0;
  for (const s of states) if (s.due <= today) n++;
  return n;
}

// 讓 Node（測試）與瀏覽器（App）都能用這個檔案
if (typeof module !== "undefined") module.exports = { schedule, INTERVALS, countDue };
"""
new_test = r"""const assert = require("assert");
const { countDue } = require("./srs.js");

assert.strictEqual(countDue([], 100), 0);                                // 空陣列
const states = [{ due: 99 }, { due: 100 }, { due: 101 }];
const before = JSON.stringify(states);
assert.strictEqual(countDue(states, 100), 2);                            // 過去 + 今天，未來不算
assert.strictEqual(countDue([{ due: 100 }], 100), 1);                    // 今天到期要算（<=）
assert.strictEqual(countDue([{ due: 101 }, { due: 102 }], 100), 0);      // 全部未到期
assert.strictEqual(JSON.stringify(states), before);                      // 不改輸入
assert.strictEqual(countDue(states, 100), countDue(states, 100));        // 同輸入重跑一致
console.log("6 countDue tests passed ✓");
"""
(APP/'srs.js').write_text(updated_srs, encoding='utf-8')
(APP/'test_count_due.js').write_text(new_test, encoding='utf-8')
for name in ['test_srs.js','test_csv.js','check_words.js','test_count_due.js']:
    run_node(name)


## 6｜獨立驗收
此格由授課端提供的合約測試驗收，不依賴 AI 自己寫的測試。預期三個字各出現一次，countDue 接受 today=100 且包含今天。

In [ ]:
(APP/'e2_contract.js').write_text("const assert=require('assert');\nconst w=require('./words.js');\nfor(const en of ['loop','variable','debug']) assert.strictEqual(w.filter(x=>x.en===en).length,1);\nconst {countDue,schedule}=require('./srs.js');\nassert.strictEqual(typeof countDue,'function');\nassert.strictEqual(countDue([],100),0);\nconst a=[{due:99},{due:100},{due:101}]; const b=JSON.stringify(a);\nassert.strictEqual(countDue(a,100),2);\nassert.strictEqual(countDue(a,100),2);\nassert.strictEqual(countDue([{due:101}],100),0);\nassert.strictEqual(JSON.stringify(a),b);\nassert.deepStrictEqual(schedule(1,true,100),{box:2,due:102});\nconsole.log('E2 contract passed');",encoding='utf-8')
run_node('e2_contract.js')

## 7｜diff 與人工 review
輸出貼給另一個乾淨對話做 review。先記下自己的發現，再讀 AI 結果。

In [ ]:
import difflib
baseline = zipfile.ZipFile(io.BytesIO(base64.b64decode('UEsDBBQAAAAIAMEBUF2XUPcx9QAAAEEBAAASAAAAYXBwL2NoZWNrX3dvcmRzLmpzS87PKy5RKM8vSilWsFUoSi0szSxK1VDS0wcL6WUVK2lacyWDFRWnpuYB1eSllisEp5ZoAMXT8osUNCCS5Qr5aRBjNKu5FBQy0xQ0FMv1gBpqahSAjKoMTYWSjKL8crB216Ki/CINpWdrWp7s7X2+Z9ezPf3v98xSUtBW8Ar299MrLinKzEvPTKvUKNcE2gIxDWS7XkZisQbIVE0spr1s732xpP3ptHVP106HmgZWCjIArDkxJUUDKlIL9lJ+TqpeTn66hhLMrwrPVi16Nn/py4ZZLxv73u/pedq6UQFsDlg+JzUvvSQDyFVSAFqh8GjOZFDYAABQSwMEFAAAAAgAwQFQXd0bo6SzAQAA/AEAAAoAAABhcHAvc3JzLmpzTZBdaxNBFIbv91e89qrBJWnUVkkIkgsvhBjEijelF0l2QtY2M8t+QEqorK2QlFgidiMmVdCosR9iFQoGuvkz2v262r/gWSJFGIaZM2fO+7xvJgNDN9JPDfy2HVoIzh2vM/PcfuyOLqf7wfhHMDiN3o//2M/pGnZOvF8/g3c9f7gTvXobuy/9Pfvy4ovXP6Pdn34LjydSJoPY7ZYqVdzGqqar3MTN2D18WCqWEU56ULnCWumG2dxE9GEnOD0Ljy7Cr59id0+qCW6YuF9+fO/Rk2JpFQWsLcnIyrgh45aMO3ReWc8DII3g8DWyz5YRjF5Ebw6ikeN9PvYHU5IOzseeO8ASEgPOUTJYqlu8ZqqCw6g1mGJtssWqaMkQGzJMoVS2Um0JmMvzKumKDdzFg4rZSDdVnvTiesKxnEIO2Tz16sy0dI426C1Hf2QoFsvNh1HvlYc1Xl3Hdl7alpJgwu8HKAuFEeU8LYILu53A7oeTmTck1m5R06gY7c7C3RnRR/bQs3v+ieN/7EpqHYvmlsZEHU2RuMC1QgELFiVaVzlTFlL/6mnW0oRuGuSkfWVZ/i9ZIvoLUEsDBBQAAAAIAMEBUF0CXTQnggEAABECAAAJAAAAYXBwL3N3LmpzdZFPSwJBGMbv+yle9rQLuf6BTmIgInjwFtEhOqyzY7uxzpozrkkGURRqhSAllQZB0CWK8BBGRt8l3NWbX6GZVcEOzWXeYZ73md+8D3IIZZBKpjJpSIBcZcWQ6yA9VzaMasiNyXEQKxyGrJ6DaAT8q49xoz4d3rlR+DlrgxubDi8m9ab3/ey1On7j0u+d+K2u33mTUGC9nklns9x6S9bC8grIFjHwvmaygi1OFadkUG2XipqWFhWi7rwq6MTKY8r40SFBO3fVimRH3o4vY3nNB5h1TYfdhQYmT+eQDkViq5JEsZ3XdMNIu5iwrEUZJrikcBrKdFugYEiswYEEgLWKbrENwixbQToyMdWcIiZKMCNVYyavkRAj4Ze0bSX4oqqq8dmkvNfG5PHevz32Tuvey81ocDT6/JIO1fi/FHnMkPmXoYRp0SHGpsVMRRIRzFEKOpcq4n6vzOcy5zEtJlrFVqtBYLekUQMqDnPdmsXEI/P7bb/X4HmNBs3xe58/wfkE4y9QSwMEFAAAAAgAwQFQXStSpxW/AAAABAEAABEAAABhcHAvbWFuaWZlc3QuanNvbqvmUlBQykvMTVWyUlCKDAlQCMtPTnQqTUmpVHjR3PN02rqna6c/XbL0yZ4tSjogpcUZ+UUl8TANcMVQyZJEoGRpUQ5ITg8ilpJZXJCTWAkSAUrnpSTm5OelQqSSEpOz04vyS/NS4pPzc/KLQGqU00zSzNJMIQpKMlJzU5HkDNLMzcygmjOT8/OKgaLR1UCDi5JB8iAhvYK8dCUdoFBmVSpIWsnU0KgCiEFiJZUFYFdn5iamp+qDFCrUxnLVcgEAUEsDBBQAAAAIAMEBUF0Qct062RkAAEA3AAAOAAAAYXBwL2luZGV4Lmh0bWylW3tTFEm2/59PkdPune0O6abBxyoPbygyO97V0avuTmwYExNFd0HX0t3VVlWDyBrRqMhb0AGRhwr4wgcCjg9E0O8y01XV/Zdf4f5OZlV1dQMzs3HZjemqrMyTJ8/jd06eTBu/CoerGDtz8uh3X7bmTkqtrDb6ZXPGnt4sLCwVBvrtyUXr8fIvuQn8/2zL0eOnWlh+a878qdccufNrrncfMwduFMfvsuNySsW76G6N3LIGVr9sDph3XpnLU+ZIH2s+9w9mjqyYfY+/bA5iQmty2ro3jpkShpHR62tqjK5wt5GJtCtGItsaUdSaTjUmtWbj8e6aPTVJqZXYGuisi+BnpC5adzAcPRyOHgIxvJt9T81XH+y+p2x/5AAD10wshBXnezGXvbhReD5qTT225h5gRj66Nhqu3Q+K9sQa++f5Myy//rHw6YnoaX4aMR+8LyyMCFYLb98XP97HQCJ6mFkD49b9ucKj/sLDLXNglYE3XTAYbtekeFYyFDVd82VzNqZEulNJZk18YCnJ0JRLzBwfsaavWusvaVD0MOeUj6zCNKw2whRdz8rM7Htmbr7FhHvqDv+aG6kUO40icbNC38/W3Mtfc6MQ/J59UXQtF//MTuJHdywb3Zm58sJa77MezxSfTVkT76zFe/mP77DkX3NXiZ26CDtzltWic1yN6STlgRvWxIpguIbaajRZiqdkfIolVV3WGfjlCpmh8fgTnEdScQbd5NevWa/e2m/HrafDwp7y6zlz7HHh2Zr5aia/Ras4qXTKzH732ryX+7I5bA08N+fu4KH5BDM/bdkv5tHFHHpQuLYFxrE4vFpDucLCTxCJdf8dCJPNzc6aa4/YKVlLSUrc4UTQKc6s2U97xbR8ob2Qnzm3KrSdXx/Ef9F4VsWypAyeIPNCrg/UsXbzPcbMkg1YI/3m5qQ18cm602/NvXDkhb/jLadOY7XkP44tQt3FOz/BlSDsYm4c2rCXPhPhz/fMW1uY0RqexWtxfty+sSbU5S0tv3G78GwSfczJsdIcQihMj2mynNYTqhHOJNS0HMmk22E4A/mPj+3rC8X5Zdac0NSUkk0x5Qx1YIUnd8yxh/bEoj0BLxgRBm+OrVhjs/bcA+vZA3vyRfHeojPTPq79OsizTZaMSu1TW1aTa2J6Z1hJZVTN8BnBvqjwSth54dkT8MQ9ybUJDIn8S2f2mwmzf8vcHGMZSdNl2GfQkC8ZIUx47PQprL757MlvSN2bk4XpB+Y4pDeFB4KYZxtCTeC92LsO2WK1mBFacqbolJJKXDLk71Utrgc1tUuvZvIlRTeUdDtNYL3+bE8ssVhCjnX82EWdiKPCkzFzsB+MWi+vwxrz66OYAJORlTyfAMSQkU0tWnODjlvd/Fjshy+NQK9gyJy9b86vldwHf4asGz8664UYYALEdd+A/aYPEGO/m4c9wbQL/c/N4cnC+1ulkX8/AT7gz8JlxXxw3OLAJPvlxm1GkpLgfPxF9CG/uj8nWCkurNiPNkAclAtLVx12l++yJCA1ec5QNaldZp2tYb52rKCwOQgSsNX81ls8sO9Pnz1+DuMdbsAIjLI4/chDELhQcfSt/fQjGsG13oUlguPmo83ftnDA66zDUs9923LyJDOH5l2lE3IpMTVNxgo9tISjdQfKJAaGrYFbTJM7FbmL2ZPT5twS2WF+fYhM8MZo4cWwY7jjA2DWGbk/wgSGkXLnXtq3V83FawKJ0mpcZsISsaBK1yi8XRVrst722kscPcYHrJ8nySVHVgrPHwlLg56B95414gE2KgAEn6w7q9bkG3v8Rv7jKPhFo7MaLu6EqhusOLtov5+h7zCpiSUHRw582bp/mMFwrOV1ZyUHsJK5nFgimeLgMByzeGeY1ewcZljh9aYA95UnWCUI88hg3bxtv39D8PJygYR+OMrsp7fNq9PW2Btza80mO3e1KyRJMe/jBjsrJ2VJh3HURgj5z0tau2w0AUjTJT0djEDF78zVT2CQLOzZpLm6Qd4zOwv8MMdvWYM/AS2At+bYqJW7B7snNVLIIStdGIBeC0s5Mv5vW5r/dvLEufMcNAdETpDfmCjO92E0md3KmJkbZobUzvIfH8LV8x9zxY1NWCCkVlx4ihauZnctPCqdPHrsx7/+/cRxEXhevqREwByYIvd99bYwvoXu1vwHJzjxTIGTmW05e/bo+aPMfHIVPi7WWux9VuwbJUVwgXKzuA4BCnS2Z9dFIDbXXuY3NuyZ6+hjf74NU0Va4dhp74wIMzDqDEL20l1QwGTF/hEYnL14iy+O4MgROX7A7vvXJDKRcEAv9othR/7h8JGqxq+On24+/88zLSxhpJJ4px8gc7q9KXA5Ef5WShsBakV0xk9KNiRgHaGs0RT4+/lvwocCbnNaSslNAXI2wvAAg28achrdupS4kWiKww9jcpi/VDMlrRiKlAzrsGq5qZaIJJV0BxaWbAqkpLTSBrwLsIQmt5Xe4fZqumI+IyGn5HBMTaqab8o90ba/HDwoU19DMZLyEUrL/gEXOkZJYGONaKxq1I1u+mWsVY1397RheLhNSinJ7nq9WzfkVDirVOtSWg/rsqa0NbRKsY52Tc2m4/V72va3HWw70JCCVSvp+mhDRorHERPq6/ZnLjUQqIYROdrT9TEwJGtXMEeitofzWe9wR217dEMydLf94MGDDZwJXbks10cOH9DkFO8Vk7R4T0q6JMRXv29/FJM4U9OETMoaKjuERpeNfQfRynkpY7oNiwBQy1oYfq9k9fpa6uJYPInhUlhPSHG1qz7KiG4tkdHaW6VgtJr+F4keCvlYrAODDVkIBwJKyjGjPo3sgHOckqG0dHv5in0jayN8bEpJhxOy0p4w0HJQrLY1axhqusdZHhbFiImShGujtLQ68O0jR8TEyjgPFas85F8k80vEYcxhk+QTy2o6njOq4ipOMBTpkrR0j39sK/SwT/L1aCeA7qkUeLkIHCZrsQZdRW7B/NaA2NYjVFwbjf5XQ6XGXUnVldQfNtRMvV/xhxy51CvpBIzW2GZQniC2c9La2rqD4LhRKJeJuPMRLZxdka6d0tt3s81Djmm6Mti3b992fnzOkpTbjHLKLCJrmmtFrsAbaxzHbaxxgIn8l2CqttLT0VLVmGFKvCnAXS1wpLEmg6a40skbybMCR7CBaKxBk9fXMd/Aka/TrXqmQYwRnyQlwPj0BEx/aNUHDhzwTUtw0ygMhtNDgiJLyQBT07GkEutwG4KhwBGRfNnLEwh0jTViTOVwtcM3FAx0BA0tK4eAnUo8LqePIEqYn6bYL3M/7UYhrVZSaJOSOpGIJSVdB4DD8D165ue7+Y0boDe1Gz1JOWakfSQlpeWSlMok5WCJq6MnWP4TYvrjEhFXAaQZT76eiZPJB/wzeeR19DL+N6tcJokVcteQ9Jmba9hll/Hn6vsiOv6n+oPVkv4Ee/8ZlzSlsGUuE0egHCd8EjLU9vakfIL3o1X4U/UdV+GQVC95AuVO3egl89y09c4AyySlmJxQk3DbpgBSIL43zSERQiIip6svJ77eUxttkDJQT3Xh7jDyft7w9cWsajS00xY6mxEv1bRJfXidJOFO48zqGDV/rtSO4JS2ZN66/Cvyj+E8I2XU/KIStueR49+FoOQ4kfRvJsoJO5pyOfTJDbDi6dPr5/5gM6xkDKZrMUzu7Oeot2iv6KBrv/lZbFV2+HykqqaGeRUVZGjW+Euq/7gTInc7l9EQgNh+Kr30LRVnhsy+Nfv9Et8jvmZ/VQwqKHEyy+vIE+03OZ4Mzphjz5FCWnc2i7lpc2y18PCevfWKdtCDOeznocOynRsmKrzKUUHh2hbVdgytuyYmGbEEUU/KBjq3sybWc6WhCp9Yj/P+P+dOfxfhm+2gn1oECf4JpE3BADaE6BkIhdi//02j2RXGyQblUM+VqrZsOkZbDqZLnUCFHubS9pHSK0hVi0l1A2JpV9q6MXF7KFROmF3hEhEVIeYV6bBuqme9fGl++GQ+eoaM3766AeljZ07inL4uKivWVB+8w6k0TC85ZLgU4lL36bY2sIS1R11R+Bv37ioG9ApwKUQrhIBkFbs5Q0UHEAiGWNMRdkoyEpG2pKpqweOSIUfSahc+1LBDB/fLB0Jsb2nOhqouJY08LdLZel6DFJOgkSYSPcz3B2E0Yxo1KbPC5jptx93uwdoQM0d/Nj+/KGwNEiI8egZf+B090Fqqfevey9IVGqgSO1VSbgS5vCrBSRuqYD0+nSMMy6fbgl2hHvQWUtDBPfR5oSsip38gYQVLb7A+pssycuxq1qWpyEKi7EqIshiljQX1CJIR1tTUxJBvyW1KWo7DnpxWVtuAx3hWxiMXdJCzy+VSGBgSPiNKJuZov/2G6kqabGQ1cAmuuTGFvT9WWREVxZKZ6xWlEb/j4WuFv+1cIPFNU5IUia85qxtqKshlJbTjMPhHXJDDieOEF34o05RLhjdXVbikM2lS0Y3fd04xyXb3pMHb/ZOvNpLJ6olgJBLxrxB9/XYrxGLdXXFMpDj9EBtvqtyMrcChGZHgpcfRwrUtp3Iy0m++mqniHnsxK3O1i/nalCRy+WAXOUjJ/rhlNHqmwRkgQB17XnjUD0ylotnAKjCEEGPmujm6wEljg0DOlk0mYdaCuT+hAQk0qMfVWDaFLR/poSUp0+Ox7hPxoBIP+Z0gjQgqlPqnoJMwhSIilJOtIoMrl4YTDeqofDA2C05gP4WpMXP2PkMqVRygKhNCof1sQ6RVLmFQpWDdLDbGIB0IuJ7DJRRJyul2I8EdKMr54QN5VrxtaP7jEGDCen2b6pZcMEIqAiituzfxtfAEmUVvoMGl5CbSoYiSTsvat+dPnSRSIq0udXOSYJ8M0IjUtrwBmWqllDgBYcv0TAAk9MOXdyH6Q0PV7itCT0CM0+P3Od2RT54rO1//IMN8I8Jx0WcRbtbfU8nNdpYvJ3blxp3i95nx873NAMXHXQzQ/vyRanOfkFKM2m+fMC+XL1sP30ioHRUg7zoflsEhXI8QtO/d28BN8iv0RxNHebR5I0mfeiwhx7PYRHBsr2ZqR3Wl4/5yY5xXK5eG8xuPwJl3LsAnEhFBo99SVNDolzPCMxHXN4gP4R96QmkzxAcZMnFaOX6VdfDAg2+P4BdgBD4iXIayNypxjtjLFGvz68NUfs5dw7P1cgGkBRyUG4RjJY49iM3rTi6JNCe/1c+xccYcfMYCCMtlvr0XTebm28AOEc2nVU+L/JgGiM3s2TfmrU129MwJfnpF5xx0gIVwuf6u2H+reGuVqtifX5hjdHpbnB4tztxE2ugPZUJ9p04fb+G+hJw+G5fDugoHM8IHwgcCjtREtZaVf98fPkDFy+K9+/bkO3NwNL9+J//xIdXVh+8Wnj9iLeFo9C+iZGvN5YD/5viovbQGBEJKh8DOjjnzYwU//qPl7LkTp78jLuqidfvC0YPhaC2fnuaX0kZCUzNKLNwpazpJ31qaLs4vm7lNitg+iiX9AOD/JncLBVFY6PCCghupO3hWs0ts7pCRF25LnrgXdDhATOMzmprKGMFA4fVmfn0ovzVPGQUWxDCeRIFsf/muObeEL6SZ92tUOc6NFZ5sIYWFcAIhAZG/E8SJm2rWsZ2hK6WUqMM1IDpl6B8kRpyCK4VpCpajOXNhiMrjC0v8aHiEgvLSgnkfdrWCME0HV4hYRkJJdyh0xsgHUBT/dI8Z3RmZYi/yK34oVa5oOgx2U2Z8PN0G2cQlQ+IxF78RhxU31LfSh9YIJ0rBLUCj4D8pKeN9ozPCyL9UJR0MkJiqSjBHuy5raMjzD0aLFV5hDW1ZA+O0GZtYEujinTZQ4vD8OR01bOsDcVhzzwmTlob5EKrqlyr5D+fo3IFEy28fuLcODjLz5gN7cdOeXTfHRyj3GX9hzs4Wpic9weY/v7ImPoBd4WTcJ+hg8/oC5JTKyBo/XKX22v1od2THJYrXlHTpR0PtkNM671FLQpb07nSMeWYOg0k2c78Mipp4NaOCrw/XNZmQXeqSFEPIKBhwL2JIGSXi+RZUlKrprK1JyboO+6OcUdh5SjYSaryeBc6cPnc+UM3bqK4HV6x3ujCAh1BwmFQaQGeqWShiq1HDDwaq3Z6XwpiXm3S956Xex22ujk4+gNihX1xKt8uamtXDcUWTY0a4lY6BZS0sxWJYCvFCYTfAkcTzPYGe5uc+c2i+AkOI/hUxEdUt6ytTZ3fJKTUuJ+sFfFazkq7qWW20bn91GXTui0ZhQp5nkb3yziz/4YH9agoIAdMqcymhbS/ACzrI7Px2U8+ikX10/aQSuBnybuvmY3Nsyt+dPH7uGrn4fmLHP9n+0mTCjOqdX1fgrlXUswvYnGDHCrGSocFKHM3Xc8NjV34QAgwROoVKSYJAA8cOYZP8tEhEbgfAhPEHqaN/s7HTnycQeJYfXpBQRnYBJ2Bjhev4Cp8lb6Htkpd6VjORlTi5eoMTSmISkh2n7OJCt9v2R/Z9RGvnyouIMJzWBeLkh5C34U2oXWUfGnyyEFGe3zJZpoIKNl1Da8WRRYr7/DhanEXzg7RLlTmKPfEAaIiRv+SeBLwFcSUKkcTKlyU06MOdnXZChNC7AbJ/bMgxrz8jQCJKFV/cBSOF4bXC8/sFZC/rdOOGcHtz3ezf4FyAcE9Ap1O7dAxAE8A2NVAduJzwHg0l4zxfcTagT3p5uhtXu9LMnBu1Xl7/s4cjxBWvDPDEjGufErJibl7UYVlzyzdn2bFaJ/uiVAsMLq8gBhR6V6xrP0OVIoTz3CAWcVmLGOpJtUvWmiXIjDYtsSQWrAdpivJvIVcIhGhdSDa7WIumqVowIObEjo6qkOMv7NlXyHsEu27i4DMJMts/lE3A/rbVAzidyoIAJyYsj0/nfGP46FjIDvaETNWaeUEFlHXnDoR7BWzU3ForPvgsdu88raDl5T8M040EZNwP56DtgLtPLGXagoHShGUbwEY5dYRUV5I8KbCxBs2NrZrz6TJPs+ldKA1GJD7AWMR8u+XevMDLzyus54v2m0Ugy9mjfw0nFUMWF5Gs6Q0LWUH/RnFhZMcyUZcsdci6ERRpo+POvO7BF8RTni5RGtyl2tZTUV+74jssdf4csj0MHVkXh67LCXq6nKhmQH/AtbNxYzVUlSOKRLo2xAiDhAVGdDrgCEqAvZDIwWgkCzOJP7idENblYJSnozugqu+8p4SqZBiuHPb5goIHr/zch3/YwaKEcH8Doai/F1t2RKZt0eOPZo/l+CQ4+S1kwtZON3ja4sCRmqFXpCEXAvy97D8/VP+Z7XWnQE6jAxT4SHzuSnSLxyuYySVLqTyQgP2IP+IbUGnd6Tf7hq3NMSpvEkJd3Rnz8KEEe95+EVszc329OLJCtSKOLI5vGCXDJHsi/4Htc5B03IlvX5wEHbgYCHlLCZBsPt2HxrBbEDgKIdFNMro8N1vsXccz1fZ714vzfeBV4Ju1/NBedM5T4Vl0HTI3jJ1V4fMt4u7Da+unT+hPIOGiH1f/xfIAxTcOPkw+qmlSd0TR+W/wYsTRCI/A3pu7G/8Km5H9+OJ52FelLh6GX4wIVSGKb8NsLJFwen4TxkOQNz6AZ5ff7eiV4Qh1MeIpmIMXWsWA0uRtqtYCjA6qGY4VDn+OG9Fuy61sxjQZzuoUN4MBcebmMsCcjVXJu9RM6Ytzjuedd/R4UhDzqB18AN+xuTJoqOjjopf7vXReUGpp2g5poRKd3yw67ebQsBqSPL8aSbfQZu97R27FO4P27dUSea+U5Cy7TCd7m2iV/801A5rm6ijVS7mK4JCedrYBsP+vno+GNReePnS8yV07ESD73UbQ5cfDdmILeyg5HW9OKMl4sNVh+cr/JxLzC/QDXs5jji6IYGyOPy2F3t89WNnxpv8AFmvdH4ffFp9NFZYH+dGAcwF45rpX66P7cLnpwsMFYKf9akFcRKZb19dfm8t3K+pTXlApP4Hv2R4/SifuXhDxqqVfld7cAqKvKcTLz3onoKwNLrSt8Os7IS/N28PrmzJ5vA5dNFXcQ/buOXuk8T0rh6pF3HeLkeKK8ZfNe6Vz5WGqwQyOCqF606X0dv8y6YCcLxPtu54fqB0OpDmJk//gyHesQzsnmBKVVB0DKz8CUjucZtBzIcjJVvj53m6nNezrrwGdvM5ZSn1DIX+BtouyTU8Wd1ZhVKIea66uifMLX16d1ThcVx6KlFbpFmlFHdit4Qvi5s0HdPv//WvH2l+NYAp79g1tkfs34AW8dvPEHOynAhfawczmW2ty3TtwKSnREzLlqHRQuTRk9ZJvMb8mnIviznXOEXIU2DxFMV5BIoaWHor7Q3xLMYqNeWHxhT35DvkHIySiLezqKP3jAtqfHxD7c2EOSSVNbPBAh+13Ug/508f4bwSDuNJJVc14hdXQm0BbTiwe4bc7vpNSNA2aGril+dEoTrrj+2HwEgwIe+VbKE8jorCNyC/2K3gt2SQBrIOMaCtLNMozCkCps8ESzubZoEz9xeR075cowTBIMDSt+GcIohES0FXkwwFQ4LT83l1xYaXk4THuHzyWlZzFgw/x1XOwCtes9EnnSI7LqNh708u0OOS6p2PMqy+DgTZFw1ZNXLrnayub0JOse8LtJm4P/JdNYFH5z/fsyWmeL4V8p3A+KLjwg6hqbLsUsFucRb5ory7QhGMrxXtU7v41N1LsH7VfTYljDl5ZnhEnvuIsnR8KP6ezD+8fYBDPy1OkCfdohUKOczeHTgvmXjiXzpc+mjeGmC5rdD2ZCHTIGlxDXJc+8/1R8guSWsDp8j3vEWBKmqWlTqVdwv43VHqMlHXDctsVnerRAf7vGghVfZeBapxrizXi7vX/AVBLAwQUAAAACADBAVBdmfaRR/EAAADHAQAADwAAAGFwcC90ZXN0X3Nycy5qc0vOzysuUUgsLk4tKlGwVShKLSzNLErVUIKIKGlacyWDVVQrFCdnpKaU5qQq1CKr09MvLirWyyoGqeSCaNJLSU0tCC4pykwucS0sTczRgOnUMNRRKCkqTdVRMDQw0NRRABqalF9hpWCko5BSmmoFFDVSqNW0VlDQ11d4vnbK0w29z2Z2Pt8yi6C5xtjNNYGbawEyl6AxptiNMYUZY2gGd97LBU1Ahz3Z0fu0rfdpbztBo4FOSUvMKYabDTXaEO5CQ2Sfv+xZ/6xz8tPZ857PnqRgCI6B/JxUvZz8dA0lE4WS1OKSYoUCkJUpCo/mTAYFPQBQSwMEFAAAAAgAwQFQXWJftLdJAgAAigMAAAwAAABhcHAvd29yZHMuanNVkz9v00AYh/d8iiMTSCVmYKhaZUBCYqkA0YEBMVycc3Lg+MzZDm0RUgMEEhLBQJIKl1RpqNrQqq0CKW5TQ78Liv/clK+A4/hSV7I82I+f931/r08QwEtCs1rqmQb+rTeCCzitI+dww6mXx7bplHvM/OiU+57VG9t1tv4T3MP62K4swQyYB8sqxYoObo/tzYdLd+4Df7cGsJJFK6m8XpAB67zxDo793rm/931sVxMiUTQdPH7w6O4ySIMnCQBeIWUhCeUcoVjPF5JzYC2/kHTthne04f5qJl/PgYihSDSohokSMay05V8MAoBLikSEGUOGdDUiJjP8+TpVTJECKgR11lAE+G/r/v5uTIEkCYsYKTrv41vVbVa8Tx88893EE0IqCtrQOOJ0ym69FIwf3L1hPSajSKcYFXkx53PL2dyagPtdb9ANwJAK0kO0CGU+VesLMxtTiyAAp9b0qj9Gp+dsux7bSoXHDpwdczQcTtczSc0aBEtyKu/dxjGQYebWvCAhqBsUCeGSb0IFF6CsTXbBZ5aRmoezkZ2dPb+/HYXGc8sSWc1jnrx7Yvl9Mx4bzMmzTNdLzDq7/J4rVKTkjJliZJfYSTumyGEKJYlLWNNinVN2djHzhJD2wsCUIp6V2/7L7E5MQkSdqIYWvfYOOuzQjHUSMs+hkoOUEN5ut3bVkTH0YCGSzH8ivz3w27+v9iFSIpIsns3M+lZYKfF0MYElcF1fVRGRQIFkDRmBa+k0SBrBmZCwgrLJG9HzFFpRCdW14ByE52Ex8R9QSwMEFAAAAAgAwQFQXTZybopmAgAAiwgAAAwAAABhcHAvaWNvbi5wbmfrDPBz5+WS4mJgYOD19HAJYmBgYgBhDhBVLeW8loGBI8jTxTGkYs7bu7sLRRxFHAJdTk7/dPCQutjJyUkTVsZ82Hgi3flA26Itn+/Ou2ir3mT26+vbmbNu7a2uqPzF5HD++kVuhgQeRoYOIYYGjVFqGFErfl/k9tixZ29/HNOc849ff36hWlNUzq/vv/j6B0vLL4nOSRvYGW/MWyxfpsR8W6hB1/LXQ7aUBG2mDbzu9ndm8Kx5rr2A1+Lvw29n7lYxbZgWy7F4xr6pe5iizj1rbv5glSHpz8Bt+fehxZnHRkwbWu6yuSdriQuArJ5x1pW572TOOVWGzzcSC1LUSsQYZxy81/TYePllTgaHKQw+T6MY0xOj1MHunHGklenc0TN+igwMgUwFc1ezzJl4cq8XA8MiFgYFMPWAa73gwz1n3h+HCYxS79z1G/5X19TK7WW5cenp29ovQj+mq5OZSj61G7A/vW/csLeSiY6J8kFEXd1/syrON6+Pr7+/ePqBwAribbdnOx7/J0iihAQtg5ja8ejj18ovQg2taCl83WcpwXMnnsSpgvKFQScwX3TPnOsEkk3QEuey4L5sNZGBwYXzAM860YKaZyVWjDOO32tSNl64meNATpmEzOSbc7zBijnFeS2kgnc2MOikbD9YOON46ju5uHlu6x/e0ct/v3g6Q2zfP1MbxjWp/w9qLMtXZbzREMuxPf3dtc9cGd5x7Ad5LX5tV2K4PZnxRufi+mimDeKSDbyWv74rMGxXYNvQ4Fyfz8Qg8FRbYcADcZSiNvWL+fGWGlv98LkFwEYGg6ern8s6p4QmAFBLAwQUAAAACADBAVBdAOgrOaAEAAB5DQAADwAAAGFwcC90ZXN0X2Nzdi5qc51XbU/bVhT+zq848hdA8sII+wTqpA5t0qRNk1Zp/UBQ5SW3xCN1UttAR4WUaoQkw4HSJtBQNtryVhivbVcQhPJfIPfa/sRf2Ll24hIGcYKTKPb1Oec+59znPsfu6ACdaPq9sDYc+E2D02Qev2AW3rCVLWbMsMzuWdKgs9t0a44aKei98wtQY4emVs6SOXN+nN9J/cEONq311fNShqYydOefs+QTPDE/pOgnw/z46rz0svd7sNIbdLJg7c+cl7It4bii6SBpGlF1uAUqeTgkq6RNcEeE9p6KxWNISKpGcFYRhqWYHJF0cjeuRjQYu+gW6HDhc8eWjg74Ag/P87w0z2bTiB9OJ54BzcwhTrYzjSfsxQ7bHC8f5+zikjNu2MVxOrVI1yat14ZVXESoPFSLCysQISRxR1flsP7twyEp1ladAXEnEjEiWi8m2d8LIeVXScGPaK8WrUJWaBdbAPoeQ0xWSDd0inBfJrGI1g19rpsgguB6Cv0wJkLVMnjR0g3JTStR0bS/vQe8A7Omi4dsYdMf7PnRSQ1eNaTwXw1qPnBj5F3NIv/mpx+RM70///Ad/pnrh7z6++/tJ1Pl4wl3Jconf5mFon9uRBFHoyHlYn6X0gj6pFGDrdEDc7Bev7VfbYEDAFz0/nABHAQgggsAoMmi3xSty3uazfEtun5oFo/LJ9tsKusLuVUYkIcJDCUEUWD5T2xpHLc3+7NgF3OYxEbOKh0IrXVy8LzhWvcm0uKsLxWQHzS1hmpkJ+c4Vw5yuLNpZsKezzeQjyb9DoIQlQVBEFn2OZt8SWdKdVJodRzQvpXnUHW4yWIgeju5ZB4aNLVOk5NuJuWjJaRS+SDpjTQnPyJ5pKtSMzTCM9en2RQ8HrHCARad7ubM/J6n0yi1y/7QESj04ayN7BVAoVI8B0/oaxoDqj1KuSP1jyE+2A30aYa9L7gdTASiqnEVq2Ann6Kw2MaOtbEMY9gRrNVpmk2zdydm/i2EoyQ8eG+Ex8OuUu0Cbksij2RNl5UB7D9YXqJ0gyDFBuKqrEcfYClHozjASnlzew6nFbCiPS0xooOK9jVAG+4e3ozXF0kVK8l6iCpL7KCpLLOr0M5tT5Od+5WZEOrn+vQhF3iNfWGHFA+4m8alBES6PI87s8E8AvHB+nRQAy7CwAMp0Ubg1tdAApzinBRIchR3bD5fXQgQH/R8+r7sD6hE0uJKQFbCsaEI0doE3HrIYQ7v/2aVvXCJ5qgtELwqfOcV4YlSE7rzUmgnOFEAJfiqiMErIo5GayIGr4qIHeiaiF2NlaDrc1QfDlSpL3qcr+2+IeX2havyodEUE+rS+eY06WqGIPvv2ELWjyBWJs3m3qChKzR2OmctpxsliWtdlyicgxt57HKuLQoWF9zMBF1eo7vT+MzNn6rrrtTt2kcin13WUPUv9QLO43/3eOeqIDLoMwOqKgp4w0p/qA8Sp6Urx1xCIiHlPi4Yv6zIhx9gZ61H+FqPBIjCHTAcb248jlALl3deR/uxCfBCHu/Ziyc09bF8NOv2Cxy/CbeCTmPijSIew9H4QJvA35j4G5aGLyQYMAKnC8/5e8p/UEsDBBQAAAAIAMEBUF0I82GiEQUAABYKAAAKAAAAYXBwL2Nzdi5qc41W21LbVhR991ds9JBIg0cOfYTSTEvbJ5p2ykz7QGhGYx9jFSO5khwTiGdMEmOTmEszEGroTBIIl5QQaJoUGuzwL4mPJD/xC93nHMs2NEAZRrb22Vr7ttaWIxGI2jfVn214n1vAf6CPXtKdJVrKQ9/AD0BLuzS/7i3f814v0EKVVuaOK0VvcdVd33FLv7rFvePKynf9n18Df+MB6EaMjKkJZzQJ9Sd3vO1df+vQ31w7rkyHIhGoHcx4q3vw5bfffMhNNm6SZlRLDjimpQ2T40rJnc7VDtfhmhkjQOd28bu38tqdXXcPXmBYh9jODZEsQqK7l5vzN6q0vFX757F/typ8vYWtD7k7IRbRvX+/dpDzn5ZEKcWCW5x3X9yrVWeOK8v13JJffkyLU/XlBSyivlKmlUVmKeVr1TmsWHjS/CZmQue3hT9GlSSoHa75T7cQm+YeiKcQoeE/PUPflbznb71ytXb00p2dDsXTRtTRTQPsVFJ3+nWDyEm8KBMhgKhp2A7EdZKM2dALg0M9aEwSB6JpC+8lKQy/pE2HxPAmriVtws7jpgUyc9LReqUHPz4FhqgmiTHsJNDQ2cnRA/xoAh2Zx6DOAwDocZAFcMNRmJhjby9cli7DpUuNJ6ATuoYCszLBM+vkNzxQD2TxYex2PbfmvS3R/PNmU+D91ENo71IjEsE6ToVTTpcJZ/+xyS78Tffe/Re0kVs0IarMXhDKsdLnRmKh/Df7dLM9/9OQUljCrogRqqm0nZAxC9Wx9FFZUXqag8Q+hT6eJ7Of9TgeWcRJW0bDoyeUFdR+VECV8gbT4lK9vIZXGJzgIwsHfMoOoRL88qI3PwVfcNn1fd//NX4gPVEVKCl//6/65GytOoW3WJ+X3/ReLqG40AcJXn+yA8QIjyeankzKTTqnNMsmqCzZIWNOG5stM9PiMjtTLZJKalEiR346PjyKhLEbisrFIEeuW1evGxFFRU5/pUUTsmxpmTDoCvR+BhNNqnagNehJoyHnzu30EEXBbYpoKq6lSQyhtMSh89leYTJozEaIi5s/QXMogBfHg1eGVMfsNzPE6tNsIiuCG8SQWhCDXR/1GU9I7UUxzvHe8wismYIWYrjdwAXZmjHPOdvOFPZIwBNGC0aSCTBHupEgOE4YT2SHwkAsy7RsZhOcsYhmm0Z2CLKMNBtzdLrgvjrCbYosJdGRGxnTwiaw1bvMgfPPceHV71b9jUn3d9zGK8RgKx6XpY+LdmkVjeJVUi/M+M8KyCj3jwXcpsJHGJFX7LY4RZ9t0r05urvNFzuD8t78yTZHw1iiD0sQZMBWil943SLiTS2pxzSH/MgcZFY+ljem245uDLfx0tFGELgXDJKBAeLIcuADt28jWxV1VEvJGUa8AaSaMSxnVGIoJycmJCnwzBHO8qCVTcrz9SxcTgsSzDgfj9Ki9kl2dTB24ToRkO2DDybUDRK2nE6v0PWqGEKg0VI9V0aWY+NBQo6cBO5EG/pKjCosf6yb7b5smyZ4c85gM2NN67Sr7S3SgT26IF8Edn/bZe/E5TMS4EDjiYuAMIf/AcTnrCY0W8bUmq+384DNEdU2R4kYPps6FyY+DVdBErQNCLvMWosOmE3zlxLdf8UlEJxlG3sEWumJ1xG/ivS0WIylJw4wfiMvoc9A1S1NM/02iZbl6ual3koRpNSoGUsnCSePlMYfYXGsLoZLRdhVMpYyLYcRdKK5tMMnVRNuLUIG/y9QSwECFAMUAAAACADBAVBdl1D3MfUAAABBAQAAEgAAAAAAAAAAAAAAgAEAAAAAYXBwL2NoZWNrX3dvcmRzLmpzUEsBAhQDFAAAAAgAwQFQXd0bo6SzAQAA/AEAAAoAAAAAAAAAAAAAAIABJQEAAGFwcC9zcnMuanNQSwECFAMUAAAACADBAVBdAl00J4IBAAARAgAACQAAAAAAAAAAAAAAgAEAAwAAYXBwL3N3LmpzUEsBAhQDFAAAAAgAwQFQXStSpxW/AAAABAEAABEAAAAAAAAAAAAAAIABqQQAAGFwcC9tYW5pZmVzdC5qc29uUEsBAhQDFAAAAAgAwQFQXRBy3TrZGQAAQDcAAA4AAAAAAAAAAAAAAIABlwUAAGFwcC9pbmRleC5odG1sUEsBAhQDFAAAAAgAwQFQXZn2kUfxAAAAxwEAAA8AAAAAAAAAAAAAAIABnB8AAGFwcC90ZXN0X3Nycy5qc1BLAQIUAxQAAAAIAMEBUF1iX7S3SQIAAIoDAAAMAAAAAAAAAAAAAACAAbogAABhcHAvd29yZHMuanNQSwECFAMUAAAACADBAVBdNnJuimYCAACLCAAADAAAAAAAAAAAAAAAgAEtIwAAYXBwL2ljb24ucG5nUEsBAhQDFAAAAAgAwQFQXQDoKzmgBAAAeQ0AAA8AAAAAAAAAAAAAAIABvSUAAGFwcC90ZXN0X2Nzdi5qc1BLAQIUAxQAAAAIAMEBUF0I82GiEQUAABYKAAAKAAAAAAAAAAAAAACAAYoqAABhcHAvY3N2LmpzUEsFBgAAAAAKAAoAUAIAAMMvAAAAAA==')))
for name in ['words.js','srs.js']:
    old=baseline.read('app/'+name).decode().splitlines(True)
    new=(APP/name).read_text(encoding='utf-8').splitlines(True)
    print(''.join(difflib.unified_diff(old,new,fromfile='before/'+name,tofile='after/'+name)))
print((APP/'test_count_due.js').read_text(encoding='utf-8'))

## 8｜保存成果
自動判斷環境：Colab 打包後下載 ZIP；本機打包後顯示儲存路徑與檔案連結。請另行儲存 Notebook。瀏覽器 UI、PWA、localStorage 請透過本機 HTTP 伺服器手動驗收。


In [ ]:
result = Path(shutil.make_archive(str(BASE / 'e2-result'), 'zip', root_dir=APP)).resolve()
if IS_COLAB:
    print('Colab: downloading', result.name)
    colab_files.download(str(result))
else:
    print('Saved locally:', result)
    from IPython.display import display, FileLink
    display(FileLink(result.relative_to(Path.cwd()).as_posix()))
